# Week 13 Assignment: Auditing and Cleaning the Semester Dataset

This week you'll apply the data cleaning workflow independently to the semester transactions dataset.
This is the same dataset you'll use for the project in Weeks 14–16, so the quality of your cleaning work here matters.

**You will:**
1. Load and audit the transactions and merchant tables
2. Identify data quality problems across three categories
3. Build a merchant-to-category lookup dictionary
4. Fix each problem using Python
5. Verify your work and save a cleaned file

**Submission:** Push your completed notebook and `transactions_clean.csv` to your GitHub repo inside a `week13/` folder. Paste your repo link in Blackboard.

---
**Dataset files needed** (place these in `week13/data/`):
- `transactions.csv`
- `merchants.csv`
- `categories.csv`
- `accounts.csv`

---
## Part 1: Load the Data

Load all four CSV files into DataFrames and print the shape of each.
Use paths from the `data` folder: `data/transactions.csv`, etc.

In [104]:
import pandas as pd
import numpy as np

# Load all four tables and print their shapes

accounts = pd.read_csv('data/accounts.csv')
categories = pd.read_csv('data/categories.csv')
merchants = pd.read_csv('data/merchants.csv')
transactions = pd.read_csv('data/transactions.csv')


In [105]:
# Preview the first 10 rows of the transactions table

transactions.head(10)


,transaction_id,date,account_id,transaction_type,category_id,merchant_id,amount,payment_method,description
0,T0000001,2022-01-01,CHK,Income,PAY,M_PAYROLL,4259.26,Direct Deposit,Paycheck
1,T0000002,2022-01-01,CHK,Transfer,RET401,M_401K,-255.56,Payroll,401k Contribution
2,T0000003,2022-01-01,INV401,Transfer,RET401,M_401K,255.56,Payroll,401k Contribution
3,T0000617,2022-01-01,CHK,Expense,GROC,M0002,-62.29,Debit,Groceries
4,T0000473,2022-01-03,CHK,Expense,PHON,M0012,-85.00,ACH,PHON
5,T0000618,2022-01-03,CC1,Expense,GAS,M0006,-89.19,Card,Gas
6,T0000377,2022-01-03,CHK,Expense,RENT,M0014,-1200.00,ACH,RENT
7,T0000569,2022-01-04,CC1,Expense,SUBS,M0011,-20.00,ACH,SUBS
8,T0000425,2022-01-05,CHK,Expense,UTIL,M0013,-156.55,ACH,UTIL
9,T0000521,2022-01-05,CHK,Expense,INS,M0015,-165.00,ACH,INS


In [106]:
# Preview the merchants table

merchants.head()


,merchant_id,merchant_name,industry,city,is_online
0,M_PAYROLL,Payroll,Banking,Online,1
1,M_SIDE,Side Hustle,Banking,Online,1
2,M_BANKINT,Bank Interest,Banking,Online,1
3,M_CC_PAY,CC Processor,Banking,Online,1
4,M_401K,401(k) Provider,Banking,Online,1


---
## Part 2: Audit

Profile the dataset before making any changes. Your goal is to understand the shape of the data
and flag anything that looks wrong.

Run all three audit checks below, then document what you found in the markdown cell at the end of this section.
**You are looking for problems — there are some to find.**

In [107]:
# Audit check 1: Missing values
# Count missing values in each column.
# Note: empty strings in category_id won't show up as NaN — check for those separately.
# Hint: (transactions['category_id'] == '').sum()

print('Missing values per column in the transactions table:')
print(transactions.isnull().sum())

print('\nEmpty strings in category_id:')
print((transactions['category_id'] == '').sum())


Missing values per column in the transactions table:
transaction_id      0
date                0
account_id          0
transaction_type    0
category_id         3
merchant_id         0
amount              2
payment_method      0
description         0
dtype: int64

Empty strings in category_id:
0


In [108]:
# Audit check 2: Amount ranges
# Use describe() to get a summary of the amount column.
# Then break it down by category to see typical ranges for each type of transaction.
# Hint: groupby('category_id')['amount'].agg(['min', 'median', 'max', 'count'])

print('Summary of the amount column:')
print(transactions['amount'].describe())

print('\nAmount ranges by category:')
print(transactions.groupby('category_id')['amount'].agg(['min', 'median', 'max', 'count']))

# Look carefully at the output. For each expense category, ask yourself:
# does the minimum or maximum value make sense given what that category represents?

expense_transactions = transactions[transactions['transaction_type'] == 'Expense']

income_transactions = transactions[transactions['transaction_type'] == 'Income']

transfer_transactions = transactions[transactions['transaction_type'] == 'Transfer']

print('\nExpense transactions amount ranges by category:')
print(expense_transactions.groupby('category_id')['amount'].agg(['min', 'median', 'max', 'count']))

print('\nIncome transactions amount ranges by category:')
print(income_transactions.groupby('category_id')['amount'].agg(['min', 'median', 'max', 'count']))

print('\nTransfer transactions amount ranges by category:')
print(transfer_transactions.groupby('category_id')['amount'].agg(['min', 'median', 'max', 'count']))

Summary of the amount column:
count     1376.000000
mean       310.820661
std       1366.264916
min      -1671.550000
25%       -106.907500
50%        -56.405000
75%         67.390000
max      10756.080000
Name: amount, dtype: float64

Amount ranges by category:
                 min    median       max  count
category_id                                    
BONUS        6303.41  7447.460  10756.08      4
CC_PAY      -1671.55     0.000   1671.55    192
DINE         -210.00   -43.930    -15.00    126
ENT          -220.00   -40.030    -10.00     84
GAS          -106.43   -56.840     -0.41    148
GROC         -842.50   -71.020    -20.00    208
INS          -195.00  -180.000   -165.00     48
INT            21.32    39.215     56.82     16
PAY          4259.26  4615.380   5000.00    105
PHON          -95.00   -90.000    -85.00     48
RENT        -1350.00 -1272.500  -1200.00     48
RET401       -300.00     0.000    300.00    210
SIDE          313.35   487.690    759.73     41
SUBS          -26

In [ ]:
# Audit check 3: Merchant ID validity
# Compare the merchant IDs used in transactions against the merchants reference table.
# Hint: build a set of valid IDs from merchants['merchant_id'],
# then find IDs in transactions that are not in that set.

# Build a list of merchant IDs which were used in the transactions table
transaction_merchant_ids = set(transactions['merchant_id'])

print('Merchant IDs in transactions:')
print(sorted(transaction_merchant_ids))

# Build a list of valid merchant IDs from the merchants table
valid_merchant_ids = set(merchants['merchant_id'])

print('\nValid merchant IDs from merchants table:')
print(sorted(valid_merchant_ids))

# Compare the two sets to find any orphaned merchant IDs in transactions

orphaned_merchant_ids = sorted(transaction_merchant_ids - valid_merchant_ids)

print('\nOrphaned Merchant IDs in the transactions table:')
print(orphaned_merchant_ids)


**Document your audit findings here.**

Be specific — list each problem type, how many rows are affected, and what made you flag it.

- Missing or empty values:
*Through the audit we were able to notice that there were three rows which had category ids which flagged as null. Additionally, we can observe that there are 2 rows where the amount flagged as null.*
- Suspicious amounts (list each category and why you flagged it):
*The only categories that I would flag with suspicion are the GAS and GROC categories as they have suspicious min's (for GROC) and max's (for GAS). The GROC category_id has an extremely large min amount (especially when compared to the median); while the GAS category_id has an abnormal max amount, which is $.41 . The GROC max seems way too unreasonably high when compared to the median and max values indicating that there is a large outlier (and warrenting further identifying/investigation). The GAS min value is strange as it seems quite unusual to make any purchase for just 41 cents, and it would seem extremely strange to have a gas bill of only $0.41, further indicating that there is some sort of suspicious activity possibily occuring.*
- Merchant ID problems:
*Through our audit on the Merchant IDs we were able to observe that there are several merchant_ids which are only present in the transactions table and not in our merchants table. We were able to flag and observe that these orphaned merchant IDs are 'M0075', 'M0099', and 'M0150'.*

---
## Part 3: Identify

For each problem you found in Part 2, write code to isolate the specific rows.
Print enough columns to confirm you've found the right rows.

You decide how many cells to use — one per problem type makes sense.
For amount problems: you decide what counts as implausible based on what you saw in the audit.
Document your reasoning for any thresholds you choose.

In [109]:
# Identify rows with missing or empty category_id
# Hint: combine isnull() and == '' with the | operator

missing_category = transactions[transactions['category_id'].isnull() | (transactions['category_id'] == '')]

print('Transactions with missing or empty category_id:')
print(missing_category[['transaction_id', 'amount', 'transaction_type', 'category_id', 'merchant_id']])



Transactions with missing or empty category_id:
   transaction_id  amount transaction_type category_id merchant_id
12       T0000975  -59.27          Expense         NaN       M0009
80       T0000634  -83.56          Expense         NaN       M0004
83       T0000572  -20.00          Expense         NaN       M0011


In [110]:
# Identify rows with missing amount

missing_amount = transactions[transactions['amount'].isnull()]

print('Transactions with missing amount:')
print(missing_amount[['transaction_id', 'amount', 'transaction_type', 'category_id', 'merchant_id']])


Transactions with missing amount:
    transaction_id  amount transaction_type category_id merchant_id
35        T0000978     NaN          Expense        DINE       M0009
180       T0000659     NaN          Expense         GAS       M0008


Identify rows with implausible amounts

* Based on your audit findings, decide which categories have suspicious values and what thresholds make sense. Add a comment explaining your reasoning.

Example reasoning (do not use this — write your own based on what you found): 

* "Category X typically ranges from $A to $B based on the median and typical spending patterns. Values outside this range are implausible."


In [111]:
# Category GAS: Purchases at (and from) gas stations should typically be between 
# $1 and $150, though this scale is depending on the assumption that the GAS category is
# also counting all purchase at a gas station; while for our assignment we will work with
# the assumption that the GAS category ONLY counts fuel purchases as the GAS category, 
# and therefore does NOT count other purchases like snacks or car washes. With this all
# in mind, our normal scale should be between $5 and $150. Although it's possible to spend 
# more than $150 at a gas station (whether on just fuel or also including snacks, 
# car-washes, etc.), it's uncommon enough that we can comfortably set it for an upper 
# threshold. On the lower end, while it is theoritically possible to spend less than $5 at 
# a gas station for fuel, it is such a rare event that it would make for a good lower 
# threshold.

# Let's gather all the transaction in the GAS category

gas_transactions = transactions[transactions['category_id'] == 'GAS']

# Now, using our scale of $5 to $150, let's identify any transactions outside this range
# NOTE: Remember that these are expenses, so the amounts are negative values and therfore
# we need to be more concious about the signs of our thresholds.

flagged_gas_transactions = gas_transactions[(gas_transactions['amount'] > -5) | 
                                            (gas_transactions['amount'] < -150)]

print('Flagged GAS transactions with implausible amounts:')
print(flagged_gas_transactions[['transaction_id', 'amount', 'transaction_type', 
                                'category_id', 'merchant_id']])


Flagged GAS transactions with implausible amounts:
   transaction_id  amount transaction_type category_id merchant_id
13       T0000620   -0.41          Expense         GAS       M0006


In [112]:
# Category GROC: From some quick searches online about typical grocery store purchases
# compared to the count of GROC transactions in our dateset (noting the timeline of 
# 2022 - 2025), we can come up with a range of theoritically reasonable purchase amounts
# for the GROC category. Thinking about the lowerbound we can quick and easyily give it 
# an amount of $5 as anything below that for grocery store purchase would be a bit 
# uncommon and would probably warrent further investigation - while most values above it
# are nearly certain to be within the realm of reasonableness. Looking on the upper end,
# things become a little more uncertain, but looking at statistics relating to grocery 
# store spendings, we can reasonably set an upper threshold of $300. While it is 
# certainly possible to spend more than #300 at a grocery store (especially if it's a 
# large family or groceries are purchased on a less frequent basis), it is an uncommon
# enough event that we could resonably set it as an upper threshold during our 
# identification phase.

# Let's gather all the transaction from in the GROC category

groc_transactions = transactions[transactions['category_id'] == 'GROC']

# Now, using our scale that we just established of $5 to $300, we can identify and flag
# any transcations outside this range. 
# NOTE: Again remember that these are expenses, so the amounts are negative values and 
# therfore we need to be aware of the signs we are using in our theresholds.

flagged_groc_transactions = groc_transactions[(groc_transactions['amount'] > -5) |
                                              (groc_transactions['amount'] < -300)]

print('Flagged GROC transactions with implausible amounts:')
print(flagged_groc_transactions[['transaction_id', 'amount', 'transaction_type', 
                                 'category_id', 'merchant_id']])

Flagged GROC transactions with implausible amounts:
    transaction_id  amount transaction_type category_id merchant_id
121       T0000645  -842.5          Expense        GROC       M0002


In [113]:
# Identify rows with orphaned merchant IDs
# Hint: use ~isin() against your set of valid merchant IDs

orphaned_merchant_transactions = transactions[~transactions['merchant_id'].isin(valid_merchant_ids)]

print('Transactions with orphaned merchant IDs:')
print(orphaned_merchant_transactions[['transaction_id', 'amount', 'transaction_type', 'category_id', 'merchant_id']])

Transactions with orphaned merchant IDs:
    transaction_id  amount transaction_type category_id merchant_id
70        T0000983  -59.64          Expense         ENT       M0099
86        T0000986  -26.73          Expense        DINE       M0150
171       T0001003  -22.25          Expense        DINE       M0075


---
## Part 4: Build the Merchant-to-Category Lookup Dictionary

Before fixing the problems, build a dictionary that maps each `merchant_id` to its correct `category_id`.
You'll use this to impute missing categories and to fix orphaned merchant IDs.

**Most of the dictionary is provided below. Three entries are missing — fill them in.**
Look at the merchants table and the categories table to figure out the correct category_id for each.

In [114]:
# Review the merchants and categories tables before filling in the dictionary
print('Merchants table:')
print(merchants.to_string())
print('\nCategories table:')
print(categories.to_string())

Merchants table:
   merchant_id        merchant_name            industry           city  is_online
0    M_PAYROLL              Payroll             Banking         Online          1
1       M_SIDE          Side Hustle             Banking         Online          1
2    M_BANKINT        Bank Interest             Banking         Online          1
3     M_CC_PAY         CC Processor             Banking         Online          1
4       M_401K      401(k) Provider             Banking         Online          1
5        M0001            FreshMart           Groceries     Neenah, WI          0
6        M0002            FreshMart           Groceries  Green Bay, WI          0
7        M0003            QuickStop           Groceries    Oshkosh, WI          0
8        M0004            QuickStop           Groceries   Kimberly, WI          0
9        M0005            BrightGas  Gas/Transportation   Appleton, WI          0
10       M0006            BrightGas  Gas/Transportation    Menasha, WI          0

In [115]:
# Merchant-to-category lookup dictionary
# Fill in the three entries marked with '????' below.

merchant_to_category = {
    # Groceries
    'M0001': 'GROC',  # FreshMart - Neenah
    'M0002': 'GROC',  # FreshMart - Green Bay
    'M0003': 'GROC',  # QuickStop - Oshkosh
    'M0004': 'GROC',  # QuickStop - Kimberly
    # Gas / Transportation
    'M0005': 'GAS',   # BrightGas - Appleton
    'M0006': 'GAS',   # BrightGas - Menasha
    'M0007': 'GAS',   # BrightGas - Kimberly
    'M0008': 'GAS',  # MetroTransit - Appleton  <-- FILL THIS IN
    # Dining
    'M0009': 'DINE',  # BrewHouse - Neenah
    'M0010': 'DINE',  # BrewHouse - Appleton
    # Subscriptions
    'M0011': 'SUBS',  # StreamFlix
    # Phone / Internet
    'M0012': 'PHON',  # GigaNet  <-- FILL THIS IN
    # Utilities
    'M0013': 'UTIL',  # CityPower
    # Rent
    'M0014': 'RENT',  # FoxRiver Apartments
    # Insurance
    'M0015': 'INS',   # HealthPlus
    # Health
    'M0016': 'HLTH',  # PharmaCare - Oshkosh
    'M0017': 'HLTH',  # PharmaCare - Appleton
    # Auto
    'M0018': 'AUTO',  # AutoWorks
    # Shopping
    'M0019': 'SHOP',  # StyleStreet
    # Education
    'M0020': 'EDU',  # BookNook  <-- FILL THIS IN
    # Entertainment
    'M0021': 'ENT',   # CoffeeCo - Neenah
    'M0022': 'ENT',   # CoffeeCo - Kimberly
    # System merchants
    'M_PAYROLL': 'PAY',
    'M_SIDE':    'SIDE',
    'M_BANKINT': 'INT',
    'M_CC_PAY':  'CC_PAY',
    'M_401K':    'RET401',
}

# Verify all entries are filled in before continuing
missing_entries = [k for k, v in merchant_to_category.items() if v == '????']
if missing_entries:
    print(f'Still missing: {missing_entries} — fill these in before moving to Part 5.')
else:
    print(f'Dictionary complete: {len(merchant_to_category)} entries')

Dictionary complete: 27 entries


---
## Part 5: Fix the Problems

Apply all fixes to a **copy** of the transactions DataFrame — never modify the original.
Write one fix per cell and add a comment explaining what each cell does.

Fix every problem you identified in Part 3. The order below is a reasonable sequence,
but adjust as needed based on what you found.

**Reminders:**
- Use `.loc[]` with a boolean mask for targeted replacements
- Use `.map()` to apply a dictionary lookup across a column
- Use `.fillna()` to fill missing values
- Calculate medians from **valid rows only** — exclude the rows you're about to fix

In [116]:
# Always work on a copy — never modify the original

cleaned_transactions = transactions.copy()

## Fix 1

In [117]:
# Fix 1: Impute missing/empty category_id values

# Our first fix will be to fill in the missing/empty category_id values using their
# corresponding merchant_id values and the merchant_to_category dictionary we created
# earlier.

(
cleaned_transactions.loc
    [cleaned_transactions['category_id'].isnull() |
    (cleaned_transactions['category_id'] == ''), 'category_id']
) = (
    cleaned_transactions.loc
    [cleaned_transactions['category_id'].isnull() |
    (cleaned_transactions['category_id'] == ''), 'merchant_id'].map(merchant_to_category)
    )

## Fix 2

In [118]:
# Fix 2: Address implausible amount values

# For this assignment we are going to replace the implausible amount values (and those
# that are missing amounts) with the median amount (for their respective category). 
# However, I feel that I should mention that I don't necesarily think that this might be
# the best approach for handling results like this - as it really is dependent on what
# you are planning to do with the cleaned data afterwards. If this dataset were to be 
# for a generalized study (such as a study on consumer spending habits), then I think 
# that this approach would be more reasonable; however, if this dataset were to be used
# for analyzing a specific individual(or family)'s spending habits, then this approach
# might not actually be the best as it could easily take out strange, but real, events
# which certainly should be considered in the analysis of that user.

# Nonetheless, let's first set the implausible amounts to NaN so that when we do our 
# calculations for finding the median they won't be included in the calculation, skewing
# our results.

# set a variable for implausible/flagged transactions for easier reference
flagged_transactions_ids = (
    list(flagged_gas_transactions['transaction_id']) + 
    list(flagged_groc_transactions['transaction_id'])
)


In [119]:

# Step 1.
(cleaned_transactions.loc[
    cleaned_transactions['transaction_id'].isin(
        flagged_transactions_ids
    ), 'amount']
) = np.nan


In [120]:
# Step 2.
# Now that we have set the implausible values to NaN, we can calculate the median amount
# for each category and use that to fill in the missing values in the amount column.

cleaned_category_medians = cleaned_transactions.groupby('category_id')['amount'].median()

In [121]:
# Step 3.
# now we can set the missing values in the amount column to the median for their 
# respective category.

cleaned_transactions.fillna(
    {'amount': cleaned_transactions['category_id'].map(cleaned_category_medians)}
    , inplace=True)

,transaction_id,date,account_id,transaction_type,category_id,merchant_id,amount,payment_method,description
0,T0000001,2022-01-01,CHK,Income,PAY,M_PAYROLL,4259.26,Direct Deposit,Paycheck
1,T0000002,2022-01-01,CHK,Transfer,RET401,M_401K,-255.56,Payroll,401k Contribution
2,T0000003,2022-01-01,INV401,Transfer,RET401,M_401K,255.56,Payroll,401k Contribution
3,T0000617,2022-01-01,CHK,Expense,GROC,M0002,-62.29,Debit,Groceries
4,T0000473,2022-01-03,CHK,Expense,PHON,M0012,-85.00,ACH,PHON
...,...,...,...,...,...,...,...,...,...
1373,T0000361,2025-12-27,CHK,Income,PAY,M_PAYROLL,5000.00,Direct Deposit,Paycheck
1374,T0000362,2025-12-27,CHK,Transfer,RET401,M_401K,-300.00,Payroll,401k Contribution
1375,T0000363,2025-12-27,INV401,Transfer,RET401,M_401K,300.00,Payroll,401k Contribution
1376,T0000973,2025-12-27,CHK,Expense,GROC,M0003,-85.64,Debit,Groceries


## Fix 3

In [122]:
# Our final fix is to address the orphaned merchant IDs. As we dicussed earlier, our plan
# is to simply set the merchant_id for these transactions with orphaned merchant IDs to
# 'UNKNOWN' rather than incorrectly give a merchant credit/responsibility for a 
# transaction that they were not actually involved in.

cleaned_transactions.loc[
    ~cleaned_transactions['merchant_id'].isin(valid_merchant_ids), 
    'merchant_id'
] = 'UNKNOWN'

---
## Part 6: Verify

Re-run your checks from Part 3 on the cleaned DataFrame.
Every problem you identified and fixed should now return 0.

Write a check for each problem type you fixed and print a clear pass/fail message for each one.

In [123]:
# The first step that we should perform after doing all of our cleaning is to check for
# any remaining missing values in the cleaned transactions table to confirm that we have
# not missed addressing any missing values during our cleaning process.

# NOTE: This check is especially useful for confirming that we have made proper changes
# during the code creation stage of completing our fixes

print('Missing values per column in the transactions table:')
print(cleaned_transactions.isnull().sum())

Missing values per column in the transactions table:
transaction_id      0
date                0
account_id          0
transaction_type    0
category_id         0
merchant_id         0
amount              0
payment_method      0
description         0
dtype: int64


In [124]:
# Another quick initial check that we can do is to see how the shape of our transaction
# dataset has changed after our cleaning steps. After running all of the steps for the
# fixes, we should see that the shape of our cleaned transactions dataset is the same as
# the original transactions dataset, confirming that we have not accidentally dropped or
# added any rows during our cleaning process.

print(f'Original shape: {transactions.shape}')
print(f'Cleaned shape: {cleaned_transactions.shape}')

Original shape: (1378, 9)
Cleaned shape: (1378, 9)


### Fix 1 Verification

In [125]:
# This verification step is specific for confirming if our fix 1 has worked as intended.

# If everything has worked as intended, then there should be o missing values after the
# imputation. Additionally, it should be noted that we originally had 3 missing 
# category_id values - so at the starting of the cleaning process we would expect this 
# to result/print the number 3, while after the imputation/cleaning process we would
# expect it to print 0.

category_missing = cleaned_transactions['category_id'].isnull().sum()
print(f'After imputation, missing category_id values: {category_missing} (expected: 0)')

After imputation, missing category_id values: 0 (expected: 0)


### Fix 2 Verification

In [126]:
# This verification step if specific for confirming if our fix 2 has worked as intended.

# If everything has worked as intended, then after cleaning this should print an empty 
# dataset. At the start of the cleaning process, this should print transactions with 
# missing amounts.
# NOTE: The great strength of this verifcation/check is that is can be used during the
# code creation stage to confirm that the changes you are making are actually having the
# intended effect on the data. For example, by running this check after running Step 1.
# we should see that the list grows to include the transactions with implausible amounts 
# that we set to NaN. 

print(cleaned_transactions[cleaned_transactions['amount'].isnull()]
      [['transaction_id', 'amount', 'transaction_type', 'category_id', 'merchant_id']])

Empty DataFrame
Columns: [transaction_id, amount, transaction_type, category_id, merchant_id]
Index: []


In [127]:
# This verification step is specific for confirming if our fix 2 has worked as intended.

# Running this code after running all of the steps for fix 1 and 2 should show the 
# changes in the category medians after removing the implausible values. While I think 
# that this check is not very useful for confirming the change in median values, as 
# these values are not strongly effected by the implausible values (as say the average
# would be), it is still a good check to run to confirm that there has been a change.
# Additionally, this check is also useful for further confirming that the category
# median has been properly calculated.

category_median_comparison = (
    pd.concat([(transactions.groupby('category_id')['amount'].median()),
               cleaned_category_medians], axis = 1, keys = ['Before', 'After']))

print('Category medians before and after fixing implausible amounts:')
print(category_median_comparison)


Category medians before and after fixing implausible amounts:
               Before     After
category_id                    
BONUS        7447.460  7447.460
CC_PAY          0.000     0.000
DINE          -43.930   -44.120
ENT           -40.030   -40.030
GAS           -56.840   -56.840
GROC          -71.020   -71.020
INS          -180.000  -180.000
INT            39.215    39.215
PAY          4615.380  4615.380
PHON          -90.000   -90.000
RENT        -1272.500 -1272.500
RET401          0.000     0.000
SIDE          487.690   487.690
SUBS          -24.000   -23.000
UTIL         -135.940  -135.940


In [128]:
# This verification step is specific for confirming if our fix 2 has worked as intended.

# After running all of the steps for fix 1 and 2, this check should show us that the 
# values that were flagged as implausible have now been filled with the median for their
# respective category. This can be seen by comparing the values of the before and after,
# with the before showing us the implausible values that we flagged and the after showing
# us what we set the values to, and the expected showing us what the median for their 
# respective category is (which should match the after values). As I mentioned, if 
# everything has worked as intended, then the after values should match the expected 
# values.

flagged_amount_before = (
    transactions[transactions['transaction_id']
      .isin(flagged_transactions_ids)]
      .groupby('category_id')['amount'].median()
)
                  

flagged_amount_after = (
    cleaned_transactions[cleaned_transactions['transaction_id']
      .isin(flagged_transactions_ids)]
      .groupby('category_id')['amount'].median()
)

expected_flagged_amount = (
    cleaned_category_medians.loc[
      transactions.loc[
          transactions['transaction_id']
              .isin(flagged_transactions_ids), 'category_id'].unique()
])

flagged_amount_comparison = (
    pd.concat([flagged_amount_before, flagged_amount_after, expected_flagged_amount],
      axis = 1, keys = ['Before', 'After', 'Expected']))

print('Flagged transactions after fixing implausible amounts:')
print(flagged_amount_comparison)


Flagged transactions after fixing implausible amounts:
             Before  After  Expected
category_id                         
GAS           -0.41 -56.84    -56.84
GROC        -842.50 -71.02    -71.02


### Fix 3 Verification

In [129]:
# This verification step is specific for confirming if our fix 3 has worked as intended.

# This check is used after running all of the steps for fixes 1 - 3 (though it could be
# run after just fix 3 as well) to confirm that there are no more orphaned merchant IDs
# in the cleaned transaction dataset. Running this code (after the fixes) should show 
# us the transaction_id's (of transactions with oprhaned merchant ID) with there old and
# new merchant_id values, allowing us to confirm that the orphaned merchant IDs have been
# properly set to there intended values. We should expect after running this check that
# the 'Before' column shows us the orphaned merchant IDs that we identified in our audit 
# and that the 'After' column shows us that these merchant IDs have been set to 'UNKNOWN'.  

orphaned_merchants_before = (
    transactions[~transactions['merchant_id']
        .isin(valid_merchant_ids)]
        [['transaction_id', 'merchant_id']]
        .set_index('transaction_id')
    )

orphaned_merchants_after = (
    cleaned_transactions[cleaned_transactions['transaction_id']
        .isin(orphaned_merchants_before.index)]
        [['transaction_id', 'merchant_id']]
        .set_index('transaction_id')
    )

orphaned_merchants_before.columns = ['Before']
orphaned_merchants_after.columns = ['After']

orphaned_comparison = orphaned_merchants_before.join(orphaned_merchants_after)

print('Orphaned merchant transactions before and after fixing orphaned merchant IDs:')
print(orphaned_comparison)


Orphaned merchant transactions before and after fixing orphaned merchant IDs:
               Before    After
transaction_id                
T0000983        M0099  UNKNOWN
T0000986        M0150  UNKNOWN
T0001003        M0075  UNKNOWN


In [130]:
# Save the cleaned transactions file

# Now that we have completed all of our cleaning steps AND have verified that our fixes
# have worked as intended, we can save the cleaned transaction dataset to a new CSV file

cleaned_transactions.to_csv('transactions_clean.csv', index=False)

print('Saved cleaned_transactions.csv')
print(f'\nFinal shape: {cleaned_transactions.shape}')

Saved cleaned_transactions.csv

Final shape: (1378, 9)


---
## Part 7: Reflection

Answer the following questions in this markdown cell.

1. **How many total problems did you find?** List each problem type, which transactions were affected, and how you identified them.

*There were three main problems that I identified through this cleaning. The first problem that I found was that were 3 entries that were missing the category_id value and 2 entries which were missing the amount value - these problems were easily found by examing the sum of columns which had null values.*
*The next problem I found was that there were several values which had amounts that were implausible - these implausible amounts were found by examing the min, median, max, and counts of categories and looking for any strange results. Through this test I found only two categories which really stood out to me (which were the GAS and GROC categories).*
*The last problem that I identified was the orphaned merchant_ids present in the transactions dataset. This was done by comparing the merchant_ids which are present in the transactions dataset, but are NOT found in the merchants dataset. Through this process we could find that there were 3 merchant_ids which are orphaned (this were 'M0075', 'M0099', and 'M0150').*

2. **For the implausible amount values**: what thresholds did you choose and why? What information from the audit led you to those numbers?

*As I already explained this earlier during the process, I will rexplain here - though some of the answer will be copied & pasted for ease of time.*

*I noticed two categories which had suspicious mins and maxs where the GAS and GROC categories - which meant that they were the only ones that warrented exploring thresholds for values.*

*When considering the GAS category we will work with the assumption that only gas/fuel itself will be purchased and categorized as fuel (so we don't need to consider snack purchases or car washes or whatever). Under this assumption it is resonable to assume that a normal scale of prices should be between $5 and $150. Although it's possible to spend more than $150 at a gas station, it's uncommon enough that we can comfortably set it for an upper threshold. On the lower end, while it is theoritically possible to spend less than $5 at a gas station for fuel, it is such a rare and improbable event that it would make for a good lower threshold.*

*When considering the GROC category there are a lot more variables which could affect the reasonableness of our threshold range, so after some quick searches online about typical grocery store purchases amounts and comparing these results with our medians and counts (making sure to note the timeline of purchases being from 2022-2025), we can come up with a range of theoritically reasonable purchase amounts for the GROC category. Thinking about the lowerbound we can quick and easyily give it an amount of $5 as anything below that for grocery store purchase would be a bit uncommon and would probably warrent further investigation - while most values above it are nearly certain to be within the realm of reasonableness. When considering the upper end, things become a little more uncertain, but looking at statistics relating to grocery store spendings, we can reasonably set an upper threshold of $300. While it is certainly possible to spend more than $300 at a grocery store (especially if it's a large family or groceries are purchased on a less frequent basis), it is an uncommon enough event that we could resonably set it as an upper threshold during our identification phase.*

3. **For the missing category_id values**: why is using the merchant_to_category dictionary a more reliable fix than simply dropping those rows?

*If we were to simply drop these rows, while we no longer have to deal with the problems with them, we lose out on all of the valuable information that it provides to our dataset - as well as skewing our results. An analogy of why it is bad to simply just drop the rows instead of fixing them would be like saying that "it's better to simply sell your car with a broken window than fix the window." By using the merchant_to_category dictionary we are able to use information (which we already know) and use it to fix/fill in our broken data. In the analogy again, we are simply fixing the window so we can use the car, rather than selling it.*

4. **For the orphaned merchant IDs**: you assigned a valid merchant based on the transaction's category. What is one limitation of this approach?

*This was not the approach that I took, though I did consider this path and ultimately decided against it, so with that in mindI feel that I can explain the limitation of this approach greatly. The main reason that I did not feel that assigning a valid merchant ID to the orphaned merchant IDs based on the transaction's category would be the best approach was simply due to the fact that I felt it would be better to label the categories as 'UNKNOWN' rather than incorrectly give a merchant responsibility for a transaction that they were not actually involved in. Again, knowing the context of what the dataset would be used for would allow us to make much better/educated decisions on matters like this - as if for example, we were going to use this dataset simply to track a individual or family's spending habits so that they can address there finicial wellness, then knowing the exact merchant_id is less relevant and having a filled in value could provide much more power. However, if these results were to be used in perhaps national record/study for finical habits or an investigation by a law/governmental office for fraud/tax crimes it might be better to simply label this factors as what they truly are, unknown. With these things in mind, I felt that the best approach would to be to label them UNKNOWN as it is much easier later down the line to change away from this approach - rather than it is revert the changes back after changing them to a known merchant ID.*

5. **Looking ahead to the project**: what would go wrong in a running balance calculation if you used the uncleaned data instead of your cleaned version?

*If we were to try and calculate a running balance based on our uncleaned data rather than our cleaned verison we would introduce several problems. The main problem that is introduced by performing this calculation with uncleaned data is that there were several results in the uncleaned dataset which didn't match reality (mainly the NaN values for amount). By trying to generate a running balance without these values our balance total will not be as close to the actual total as we could be with the cleaned verison. It is important to state that while our clean version of the dataset is also not likely to match exactly with the actual real running balance, it will be much closer to the actual balance and would be calculated using much more sound methods, leading us to be more confident in its result/answer.*